# StateMachine — practical example

Companion to [`state_machine.md`](state_machine.md). Demonstrates a single connection lifecycle and how `StateMachine` rejects illegal transitions.

No external infrastructure is required — the notebook only uses `wattleflow.concrete.state_machine.StateMachine` and Python's `enum`.

## 1. Define states and actions

We model a simplified connection lifecycle: `NEW → CONNECTING → CONNECTED → DISCONNECTED`, with a possible transition to `FAILED`.

In [ ]:
from enum import Enum
from wattleflow.concrete.state_machine import StateMachine


class ConnState(Enum):
    NEW = "new"
    CONNECTING = "connecting"
    CONNECTED = "connected"
    FAILED = "failed"
    DISCONNECTED = "disconnected"


class ConnAction(Enum):
    CONNECT = "connect"
    CONNECT_OK = "connect_ok"
    CONNECT_FAIL = "connect_fail"
    DISCONNECT = "disconnect"

## 2. Transition table

Key: `(current_state, action)`. Value: new state. Any combination not in the table is **forbidden**.

In [ ]:
TRANSITIONS = {
    (ConnState.NEW,         ConnAction.CONNECT):      ConnState.CONNECTING,
    (ConnState.CONNECTING,  ConnAction.CONNECT_OK):   ConnState.CONNECTED,
    (ConnState.CONNECTING,  ConnAction.CONNECT_FAIL): ConnState.FAILED,
    (ConnState.CONNECTED,   ConnAction.DISCONNECT):   ConnState.DISCONNECTED,
}

## 3. Happy path

In [ ]:
fsm = StateMachine(TRANSITIONS, initial=ConnState.NEW, name="DemoConn")
print(fsm, "(initial)")

fsm.apply(ConnAction.CONNECT)
print(fsm, "after CONNECT")

fsm.apply(ConnAction.CONNECT_OK)
print(fsm, "after CONNECT_OK")

fsm.apply(ConnAction.DISCONNECT)
print(fsm, "after DISCONNECT")

## 4. Alternative path — connection failure

In [ ]:
fsm = StateMachine(TRANSITIONS, initial=ConnState.NEW, name="FailingConn")
fsm.apply(ConnAction.CONNECT)
fsm.apply(ConnAction.CONNECT_FAIL)
print(fsm, "connection is in FAILED")

## 5. Illegal transition

`DISCONNECT` from `FAILED` is not in the table, so `apply()` raises `ValueError`.

In [ ]:
try:
    fsm.apply(ConnAction.DISCONNECT)
except ValueError as e:
    print("Caught:", e)

## 6. Side-effect-free query (`can`)

`can()` does not mutate state — safe for UI / log / polling code.

In [ ]:
fsm = StateMachine(TRANSITIONS, initial=ConnState.NEW)
for action in ConnAction:
    print(f"can({action.name})", "=>", fsm.can(action))

## 7. Summary

- The transition table is a single source of truth for the lifecycle
- `can()` is safe for polling; `apply()` enforces the change
- `Generic[State, Action]` catches mismatched enums at type-check time
- Components hold the FSM by composition (`self._fsm = StateMachine(...)`), not by inheritance